<a href="https://colab.research.google.com/github/RiyaKhushiRadha/CodecTechnologyInternshipProject/blob/main/Sentiment_Analysis_on_Twitter_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Load the Data**

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("crowdflower/twitter-airline-sentiment")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/twitter-airline-sentiment


In [ ]:
import pandas as pd
df = pd.read_csv('/kaggle/input/twitter-airline-sentiment/Tweets.csv')
print(df.head())

# Keep relevant columns
df = df[['tweet_id','text', 'airline_sentiment']]

             tweet_id airline_sentiment  airline_sentiment_confidence  \
0  570306133677760513           neutral                        1.0000   
1  570301130888122368          positive                        0.3486   
2  570301083672813571           neutral                        0.6837   
3  570301031407624196          negative                        1.0000   
4  570300817074462722          negative                        1.0000   

  negativereason  negativereason_confidence         airline  \
0            NaN                        NaN  Virgin America   
1            NaN                     0.0000  Virgin America   
2            NaN                        NaN  Virgin America   
3     Bad Flight                     0.7033  Virgin America   
4     Can't Tell                     1.0000  Virgin America   

  airline_sentiment_gold        name negativereason_gold  retweet_count  \
0                    NaN     cairdin                 NaN              0   
1                    NaN    jnar

**Preprocess the Data**

In [ ]:
import pandas as pd
import numpy as np

# Map sentiment labels to numbers
label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['airline_sentiment'].map(label_map)


In [ ]:
# Clean text (remove URLs, mentions, etc.)
import re
def clean_text(text):
    text = re.sub(r'http\S+|@\S+|#\S+|[^A-Za-z0-9\s]+', '', text)
    return text.lower()

df['clean_text'] = df['text'].apply(clean_text)

In [ ]:
# Split data

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)
print('XTest\n',X_test,'\nXTrain\n', X_train)
print('YTest\n',y_test,'\nYtrain\n', y_train)

XTest
 2998                                                  past
7719      would you say a delay is more likely thanks s...
8575      i cheated on you and im sorry ill never do it...
618       disappointed that u didnt honor my 100 credit...
11741     the airline is embarrassing itself i get that...
                               ...                        
632       but its hard to stay upset at someone when th...
8982      will you be issuing a travel advisory for clt...
6329      next once we arrived to atlanta we waited a f...
1179      looks like today will be my 6th consecutive d...
6553      my flight was cancelled flightled monday  jus...
Name: clean_text, Length: 2928, dtype: object 
XTrain
 1262                                    what would it cost
10772     used 2 get emails 1 prepurchase a snack and 2...
4204      no it was 2 flight cancelled flightlations on...
5491        not frustrated just an idea great crew thanks 
12096         narrowly made standbylots of snags this

**Tokenize and Pad Sequences**

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_words = 20000  # Max vocabulary size
max_len = 40       # Max tweet length

tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

**Prepare GloVe Embeddings**

In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2025-07-12 15:50:42--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-07-12 15:50:42--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-07-12 15:50:42--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [ ]:
embedding_dim = 100
embeddings_index = {}

with open('glove.6B.100d.txt', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

word_index = tokenizer.word_index
num_words = min(max_words, len(word_index) + 1)
embedding_matrix = np.zeros((num_words, embedding_dim))

for word, i in word_index.items():
    if i >= max_words:
        continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

**Handle Class Imbalance**

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.utils import to_categorical

y_train_cat = to_categorical(y_train, num_classes=3)
y_test_cat = to_categorical(y_test, num_classes=3)

class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weights_dict = dict(enumerate(class_weights))

**Build and Train the Bi-LSTM Model**

In [ ]:
from keras.models import Sequential
from keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from keras.utils import to_categorical

# Convert labels to categorical
y_train_cat = to_categorical(y_train, num_classes=3)
y_test_cat = to_categorical(y_test, num_classes=3)

model = Sequential()
model.add(Embedding(input_dim=num_words,
                    output_dim=embedding_dim,
                    weights=[embedding_matrix],
                    input_length=max_len,
                    trainable=False))
model.add(Bidirectional(LSTM(128, dropout=0.2, recurrent_dropout=0.2)))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(3, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(X_train_pad, y_train_cat,
                    batch_size=128,
                    epochs=10,
                    validation_split=0.1,
                    verbose=1)

Epoch 1/10


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


83/83 ━━━━━━━━━━━━━━━━━━━━ 47s 469ms/step - accuracy: 0.6205 - loss: 0.8809 - val_accuracy: 0.7125 - val_loss: 0.7438
Epoch 2/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 40s 459ms/step - accuracy: 0.7191 - loss: 0.7103 - val_accuracy: 0.7483 - val_loss: 0.6242
Epoch 3/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 38s 428ms/step - accuracy: 0.7391 - loss: 0.6641 - val_accuracy: 0.7423 - val_loss: 0.6039
Epoch 4/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 44s 460ms/step - accuracy: 0.7440 - loss: 0.6408 - val_accuracy: 0.7662 - val_loss: 0.5712
Epoch 5/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 39s 433ms/step - accuracy: 0.7662 - loss: 0.5957 - val_accuracy: 0.7696 - val_loss: 0.5574
Epoch 6/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 42s 450ms/step - accuracy: 0.7651 - loss: 0.5835 - val_accuracy: 0.7671 - val_loss: 0.5544
Epoch 7/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 42s 464ms/step - accuracy: 0.7727 - loss: 0.5692 - val_accuracy: 0.7654 - val_loss: 0.5591
Epoch 8/10
83/83 ━━━━━━━━━━━━━━━━━━━━ 41s 462ms/step - accuracy: 0.7802 - loss: 0.5484 - val_accuracy: 0.785

**Evaluate the Model**

In [ ]:
loss, accuracy = model.evaluate(X_test_pad, y_test_cat, verbose=0)
print(f'Test Accuracy: {accuracy:.4f}')

Test Accuracy: 0.7783


In [ ]:
# 11. Classification Report and Confusion Matrix
from sklearn.metrics import classification_report, confusion_matrix
y_pred = model.predict(X_test_pad)
y_pred_labels = np.argmax(y_pred, axis=1)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_labels, target_names=['negative', 'neutral', 'positive']))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_labels))

92/92 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step

Classification Report:

              precision    recall  f1-score   support

    negative       0.85      0.87      0.86      1835
     neutral       0.64      0.56      0.60       620
    positive       0.68      0.71      0.69       473

    accuracy                           0.78      2928
   macro avg       0.72      0.71      0.72      2928
weighted avg       0.77      0.78      0.78      2928


Confusion Matrix:

[[1596  152   87]
 [ 202  347   71]
 [  90   47  336]]


**Prepare the Text Data**

In [ ]:
# Example: Predict sentiment for a list of new tweets
new_tweets = [
    "Flight was delayed but the staff was friendly.",
    "Absolutely terrible experience with the airline.",
    "Everything went smoothly, thank you!"
]

# Use your cleaning function
cleaned_new = [clean_text(tweet) for tweet in new_tweets]

# Tokenize and pad
new_seq = tokenizer.texts_to_sequences(cleaned_new)
new_pad = pad_sequences(new_seq, maxlen=40)  # Use the same maxlen as training

**Make Predictions**

In [ ]:
pred_probs = model.predict(new_pad)  # Shape: (num_samples, 3)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step


**Interpret the Results**

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, confusion_matrix, classification_report

# Example predicted probabilities (from your model)
# pred_probs = model.predict(new_tweets) → assuming already done
pred_classes = np.argmax(pred_probs, axis=1)  # e.g., [1, 0, 2]

# Label mapping
label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
pred_labels = [label_map[i] for i in pred_classes]

# Example: true class labels for new_tweets (ground truth, should be same length as new_tweets)
# Replace this with actual labels
true_classes = [1, 0, 2]  # e.g., [1, 0, 2]
true_labels = [label_map[i] for i in true_classes]

# Print predictions
for tweet, label in zip(new_tweets, pred_labels):
    print(f"Tweet: {tweet}\nPredicted sentiment: {label}\n")

# Compute and print F1-score
f1 = f1_score(true_classes, pred_classes, average='weighted')
print(f"F1 Score (weighted): {f1:.4f}\n")

# Compute and print confusion matrix
cm = confusion_matrix(true_classes, pred_classes)
print("Confusion Matrix:")
print(cm)

# Optional: Classification report for precision/recall/F1 per class
print("\nClassification Report:")
print(classification_report(true_classes, pred_classes, target_names=['negative', 'neutral', 'positive']))


Tweet: Flight was delayed but the staff was friendly.
Predicted sentiment: negative

Tweet: Absolutely terrible experience with the airline.
Predicted sentiment: negative

Tweet: Everything went smoothly, thank you!
Predicted sentiment: positive

F1 Score (weighted): 0.5556

Confusion Matrix:
[[1 0 0]
 [1 0 0]
 [0 0 1]]

Classification Report:
              precision    recall  f1-score   support

    negative       0.50      1.00      0.67         1
     neutral       0.00      0.00      0.00         1
    positive       1.00      1.00      1.00         1

    accuracy                           0.67         3
   macro avg       0.50      0.67      0.56         3
weighted avg       0.50      0.67      0.56         3



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
